# 02 — Retrieval + Embedding Shootout (bge-m3 vs zembed-1)
Ponytail: one file, no DB. Run on server.

In [1]:
import requests
MAILTO="ntminhduy1408@gmail.com"
q="CAN intrusion detection"
r=requests.get("https://api.openalex.org/works", params={"search":q,"per-page":5,"mailto":MAILTO}, timeout=60)
print(r.status_code)
[
 print('-',w.get('display_name'),w.get('publication_year'))
 for w in r.json().get('results',[])
]

200
- An Intrusion-Detection Model 1987
- UNSW-NB15: a comprehensive data set for network intrusion detection systems (UNSW-NB15 network data set) 2015
- Intrusion detection system based on the analysis of time intervals of CAN messages for in-vehicle network 2016
- Survey of intrusion detection systems: techniques, datasets and challenges 2019
- CANet: An Unsupervised Intrusion Detection System for High Dimensional CAN Bus Data 2020


[None, None, None, None, None]

In [2]:
# ponytail: 4 pos + 3 neg only, enough to judge separation
POS=["Knowledge distillation GNN Transformer to tiny ECU student <10K params for CAN IDS",
 "Small language models MiniLM DistilBERT convert CAN traffic to tokens for intrusion detection",
 "Open-set few-shot cross-dataset CAN IDS Car-Hacking to CIC-IoV-2024",
 "Vision Transformer on CAN image recurrence plots for intrusion detection"]
NEG=["HairCLIP text image hair editing StyleGAN",
 "AlphaFold protein 3D structure prediction",
 "LLM summarization of legal contracts"]
THESIS="deep learning for in-vehicle CAN intrusion detection"
print(len(POS),len(NEG))

4 3


In [3]:
from sentence_transformers import SentenceTransformer
import torch
def scores(name):
    m=SentenceTransformer(name, trust_remote_code=True)
    # zembed wants prefixes; harmless for bge
    qt=m.encode(['query: '+THESIS], normalize_embeddings=True)
    P=m.encode(['passage: '+t for t in POS], normalize_embeddings=True)
    N=m.encode(['passage: '+t for t in NEG], normalize_embeddings=True)
    import numpy as np
    ps=(P@qt.T).ravel(); ns=(N@qt.T).ravel()
    print(f"{name}\npos={sorted(round(float(x),3) for x in ps)}\nneg={sorted(round(float(x),3) for x in ns)}")
    print(f"gap={min(ps)-max(ns):.3f} (pos_min - neg_max, want >0.05)")
for n in ["BAAI/bge-m3","zeroentropy/zembed-1-embedding"]:
    scores(n)

/mnt/data2/ntmduy/paper-drill/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/data2/ntmduy/paper-drill/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 21101.58it/s]


BAAI/bge-m3
pos=[0.512, 0.544, 0.592, 0.654]
neg=[0.401, 0.411, 0.489]
gap=0.023 (pos_min - neg_max, want >0.05)


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 3021.99it/s]
Default prompt name is set to 'document'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


zeroentropy/zembed-1-embedding
pos=[0.396, 0.578, 0.728, 0.757]
neg=[0.198, 0.23, 0.235]
gap=0.161 (pos_min - neg_max, want >0.05)


Judge: bigger gap wins. If zembed gap > bge gap → lock `zembed-1 @1536 int8`. Paste gaps back.